In [2]:
import pandas as pd

gbm_df = pd.read_csv("/Users/alaa.mohamed/Desktop/MMContraFuse/data/1D_MRI/GBM_pyradiomic_features.csv")
lgg_df = pd.read_csv("/Users/alaa.mohamed/Desktop/MMContraFuse/data/1D_MRI/LGG_pyradiomic_features.csv")

In [ ]:
# Add tumor type labels
gbm_df['type'] = 1
lgg_df['type'] = 0

print(f"✓ GBM samples: {len(gbm_df)}")
print(f"✓ LGG samples: {len(lgg_df)}")

# Merge radiomics datasets
radiomics_df = pd.concat([gbm_df, lgg_df], ignore_index=True)
print(f"✓ Combined radiomics samples: {len(radiomics_df)}")

# # Load clinical data
# print("Loading clinical data...")
# clinical_df = pd.read_csv("/Users/alaa.mohamed/Desktop/MMContraFuse/data/processed_tabular_data/clinical.csv")
# print(f"✓ Clinical data samples: {len(clinical_df.case_id.unique())}")

# # Merge with clinical data
# merged_df = pd.merge(clinical_df, radiomics_df, left_on="case_id", right_on="PatientID" ,how='inner')

# print(f"✓ Final merged dataset: {len(merged_df)} samples")
# print(f"✓ Total features: {len(merged_df.columns)}")

# # Display basic info
# print("\nDataset Info:")
# print(f"- Shape: {merged_df.shape}")
# print(f"- Tumor type distribution:")
# print(merged_df['Tumor_Type'].value_counts())

# Save merged dataset
radiomics_df.to_csv("/Users/alaa.mohamed/Desktop/MMContraFuse/data/processed_tabular_data/radio1D_clinical.csv", index=False)


✓ GBM samples: 102
✓ LGG samples: 65
✓ Combined radiomics samples: 167


In [7]:
import pandas as pd
import numpy as np

df= radiomics_df
# 1. Check current data types
print("Current data types:")
print(df.dtypes)
print("\nData type counts:")
print(df.dtypes.value_counts())

# 2. Find object/string columns specifically
object_columns = df.select_dtypes(include=['object']).columns.tolist()
print(f"\nObject/string columns: {object_columns}")

# 3. Inspect problematic columns
for col in object_columns:
    print(f"\nColumn: {col}")
    print(f"Sample values: {df[col].dropna().head().tolist()}")
    print(f"Unique values count: {df[col].nunique()}")
    
# 4. Convert object columns to numeric (method 1 - aggressive)
for col in object_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')  # Converts non-numeric to NaN

# 5. Alternative: More careful conversion (method 2 - recommended)
def convert_to_numeric(df, exclude_cols=['case_id', 'Patient_ID']):
    """Convert all columns to numeric except specified ones"""
    df_clean = df.copy()
    
    for col in df.columns:
        if col not in exclude_cols:  # Keep ID columns as strings
            # Try to convert to numeric
            df_clean[col] = pd.to_numeric(df[col], errors='coerce')
    
    return df_clean

# Apply the conversion
df_clean = convert_to_numeric(df, exclude_cols=['case_id', 'Patient_ID'])

# 6. Handle NaN values created during conversion
print(f"\nNaN values after conversion: {df_clean.isnull().sum().sum()}")

# Fill NaN values (choose one method):
# Option A: Fill with 0
df_clean = df_clean.fillna(0)

# Option B: Fill with column median
for col in df_clean.select_dtypes(include=[np.number]).columns:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# 7. Verify final data types
print("\nFinal data types:")
print(df_clean.dtypes.value_counts())

# 8. Check for any remaining object columns
remaining_objects = df_clean.select_dtypes(include=['object']).columns.tolist()
print(f"\nRemaining object columns: {remaining_objects}")

# 9. Final check - ensure all feature columns are numeric
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumeric columns: {len(numeric_cols)}")
print(f"Total columns: {len(df_clean.columns)}")

Current data types:
PatientID                                    object
flair_diagnostics_Image-original_Mean       float64
flair_diagnostics_Image-original_Minimum    float64
flair_diagnostics_Image-original_Maximum    float64
flair_diagnostics_Mask-original_VoxelNum      int64
                                             ...   
t2_original_ngtdm_Coarseness                float64
t2_original_ngtdm_Complexity                float64
t2_original_ngtdm_Contrast                  float64
t2_original_ngtdm_Strength                  float64
Tumor_Type                                   object
Length: 450, dtype: object

Data type counts:
float64    442
int64        6
object       2
Name: count, dtype: int64

Object/string columns: ['PatientID', 'Tumor_Type']

Column: PatientID
Sample values: ['TCGA-76-6664', 'TCGA-76-6663', 'TCGA-76-6662', 'TCGA-76-6661', 'TCGA-76-6657']
Unique values count: 167

Column: Tumor_Type
Sample values: ['GBM', 'GBM', 'GBM', 'GBM', 'GBM']
Unique values count: 2

NaN 

In [ ]:
df.T